*0.4 Deep learning basics*

# Transformer: positional encoding

**The situation.** A transformer's attention scores every token against every other token with dot products — and a dot product does not care about order. "dog bites man" and "man bites dog" produce exactly the same set of attention outputs. A model that cannot tell those apart cannot read.

**Positional encoding.** Add a position-dependent pattern to each token's vector before the first layer. The original transformer used fixed sine and cosine waves of different frequencies: each position gets a unique fingerprint, and nearby positions get similar ones. GPT and BERT learn the position vectors instead; modern models (Llama) rotate the query and key vectors by position (RoPE). Same purpose: put order into vectors so attention can see it.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**The sinusoidal table, and proof that it is needed.** Run a real `nn.TransformerEncoderLayer` on a sentence and its reversed version, with and without positions.

In [2]:
import math

import torch
from torch import nn


def sinusoidal_positions(length: int, size: int) -> torch.Tensor:
    position = torch.arange(length).unsqueeze(1)
    frequency = torch.exp(torch.arange(0, size, 2) * (-math.log(10_000.0) / size))
    table = torch.zeros(length, size)
    table[:, 0::2] = torch.sin(position * frequency)  # even columns: sine
    table[:, 1::2] = torch.cos(position * frequency)  # odd columns: cosine
    return table


positions = sinusoidal_positions(length=50, size=64)
print("table shape:", tuple(positions.shape))
print(
    "position 0 vs 1 similarity:",
    f"{torch.cosine_similarity(positions[0], positions[1], dim=0):.2f}",
    "| position 0 vs 25:",
    f"{torch.cosine_similarity(positions[0], positions[25], dim=0):.2f}",
)

torch.manual_seed(0)
layer = nn.TransformerEncoderLayer(
    d_model=64, nhead=4, dim_feedforward=128, batch_first=True, dropout=0.0
).eval()
tokens = torch.randn(1, 5, 64)  # five token vectors
reversed_tokens = tokens.flip(dims=[1])

with torch.no_grad():
    without = layer(tokens)
    without_reversed = layer(reversed_tokens).flip(dims=[1])  # flip back to compare token by token
    with_pos = layer(tokens + positions[:5])
    with_pos_reversed = layer(reversed_tokens + positions[:5]).flip(dims=[1])

print(
    "no positions:   reversed sentence gives the same outputs per token →",
    torch.allclose(without, without_reversed, atol=1e-5),
)
print(
    "with positions: reversed sentence gives the same outputs per token →",
    torch.allclose(with_pos, with_pos_reversed, atol=1e-5),
)
assert torch.allclose(without, without_reversed, atol=1e-5) and not torch.allclose(
    with_pos, with_pos_reversed, atol=1e-5
)

table shape: (50, 64)
position 0 vs 1 similarity: 0.97 | position 0 vs 25: 0.63
no positions:   reversed sentence gives the same outputs per token → True
with positions: reversed sentence gives the same outputs per token → False


**Reading the output.** Without positions, reversing the sentence changes nothing — each token's output is identical, so the layer is order-blind. With positions added, the outputs differ: order is now visible. Neighbouring positions have similar fingerprints (0 vs 1 close), distant ones less so.

```
token vector   [0.2, -0.1, 0.5, …]
+ position 3   [sin(3·f₀), cos(3·f₀), sin(3·f₁), cos(3·f₁), …]
= what the first layer sees: content AND where it is
```

**The rule to remember.** Attention is a set operation; positional encoding turns the set back into a sequence. Every transformer has one — sinusoidal, learned, or rotary.

| Use it when | Don't when | Instead use |
|---|---|---|
| building or reading any transformer | — | — |

**Watch out**
- Learned positions (GPT-2, BERT) stop at the trained length — the origin of hard context limits like 512 or 1024.
- Sinusoidal and rotary encodings extend beyond training length in principle, but quality degrades; "context extension" work is about this.
- Add positions *once*, before the first layer; adding them at every layer is a bug that trains anyway.